# Merge Excel sheets

https://stackoverflow.com/questions/20908018/import-multiple-excel-files-into-python-pandas-and-concatenate-them-into-one-dat

In [110]:
# libraries
%run utilities.ipynb
start_time0 = time.time()
start_time  = time.time()
print(f'Importing libraries ...')

%run utilities.ipynb # for timediff() function
import os
from pathlib import Path
import openpyxl
import pandas as pd
from tqdm import tqdm, notebook # notebook version of tqdm

print(f'Importing libraries completed: {timediff(start_time, time.time())}', '\n')

Importing libraries ...
Importing libraries completed: 0.0sec 



In [111]:
# set paths
start_time = time.time()
print(f'Setting paths ...')

pthDl      = str(Path.home() / 'Downloads')
pthPy      = r'P:\Investment Operations\GRC\Compliance\Daily\py_reports.xlsm'
pthReports = r'\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting'
pthTest    = r'P:\Working Folders\Hilton\W\Reg_Tests'

print(f'Setting paths completed: {timediff(start_time, time.time())}', '\n')

Setting paths ...
Setting paths completed: 0.0sec 



In [112]:
# get report date from the py_report.xlsm sheet
start_time = time.time()
print(f'Getting fund codes and report date ...')

# get credentials
df      = pd.read_excel(pthPy, sheet_name = 'creds', header = None, usecols = 'A', nrows = 2)
aladdin = df.iloc[0, 0]
sesame  = df.iloc[1, 0]

# get report date
df1     = pd.read_excel(pthPy, sheet_name = 'downloader', usecols = 'D', nrows = 1).dropna()
rptDate = list(df1)[0].date() # convert dataframe header to a date

# get fund codes
df2     = pd.read_excel(pthPy, sheet_name = 'downloader', usecols = 'A').dropna()
f_codes = (',').join(df2.iloc[:,0].tolist())

print(f' {datetime.datetime.strftime(rptDate, "%A %d %B %Y")} for {len(df2)} funds: {f_codes}')

print(f'Getting fund codes and report date completed: {timediff(start_time, time.time())}', '\n')

Getting fund codes and report date ...
 Friday 31 January 2025 for 3 funds: PRPABF,PEYF,PABS
Getting fund codes and report date completed: 3.6sec 



In [113]:
# get the fund NAVs
start_time = time.time()
print(f'Getting the fund NAVs as at {rptDate.strftime("%A %d %B %Y")} ...')

when = rptDate
# def osprey(rpt_type = 'r28i', funds = 'PABS,PIMBAL', d_from as datetime, d_to as datetime, sfx = 'csv', al, xe):
osprey('fnav', f_codes, when, when, '', 'csv', aladdin, sesame)

# dataframe the fund NAVs
fund_navs = pd.read_csv(os.path.join(Path.home(), "Downloads", f'FNAV ({len(df2)}) {when.strftime("%d%b%Y")}.csv'))

print(f'Getting the fund NAVs as at {when.strftime("%A %d %B %Y")} completed: {timediff(start_time, time.time())}', '\n')

Getting the fund NAVs as at Friday 31 January 2025 ...

C:\Users\hilton.netta\Downloads\Fund Net Asset Value(2).csv -> C:\Users\hilton.netta\Downloads\FNAV (3) 31Jan2025.csv
  Roundtrip time to run the report lookup function: 38.5sec 

Getting the fund NAVs as at Friday 31 January 2025 completed: 42.6sec 



In [95]:
# list and then pick out the .xls files
start_time = time.time()
print(f'Identifying the <... lookthrough {rptDate.strftime("%d%m%Y")}.xls> files ...')

# pick out the .xls files
files_xls = [f for f in os.listdir(pthDl) if f[-25:] == f'lookthrough {datetime.datetime.strftime(rptDate, "%d%b%Y")}.xls']

print(f' Number of .xls files to be converted to .xlsx: {len(files_xls)}')
print(f' {", ".join(files_xls)}')

print(f'Identifying the <... lookthrough {rptDate.strftime("%d%m%Y")}.xls> files completed: {timediff(start_time, time.time())}', '\n')

Identifying the <... lookthrough 31012025.xls> files ...
 Number of .xls files to be converted to .xlsx: 3
 PABS lookthrough 31Jan2025.xls, PEYF lookthrough 31Jan2025.xls, PRPABF lookthrough 31Jan2025.xls
Identifying the <... lookthrough 31012025.xls> files completed: 0.0sec 



In [67]:
# convert .xls to .xlsx
start_time = time.time()
print(f'Merging {len(files_xls)} .xlsx converted files ...')

import win32com.client as win32 # library to convert xls to xlsx
# start Excel - if Excel does not open, close any Excel application already open, else restart the pc
excel = win32.gencache.EnsureDispatch('Excel.Application')

excel.DisplayAlerts = False # suppress the Excel warning dialogue

for file in notebook.tqdm(files_xls):
    wb = excel.Workbooks.Open(os.path.join(pthDl, file))            # open the .xls file
    wb.SaveAs(os.path.join(pthDl, file + 'x'), FileFormat = 51)     # FileFormat = 51/56 is for .xlsx/.xls extension
    wb.Close()                                                      # close the .xlsx file

#excel.Application.Quit()
excel.DisplayAlerts = True # unsuppress Excel warning dialogue

print(f'Merging {len(files_xls)} .xlsx converted files completed: {timediff(start_time, time.time())}', '\n')

Merging 2 .xlsx converted files ...


  0%|          | 0/2 [00:00<?, ?it/s]

Merging 2 .xlsx converted files completed: 4.1sec 



In [109]:
# =================
# merge the files by looping over them and appending to an initially empty dataframe
start_time = time.time()
print(f'Merging {len(files_xls)} files ...')

# list the .xlsx files just downloaded from Eagle
#files_xlsx = [fl + 'x' for fl in files_xls]

data = []
for f in notebook.tqdm(files_csv):
    df = pd.read_csv(os.path.join(Path.home(),'Downloads',f), sheet_name = 'Reg 28 Report - Incl Effective ')
    #print(os.path.join(Path.home(),'Downloads',f))

print(f'Merging {len(files_xls)} files completed: {timediff(start_time, time.time())}', '\n')
# ==================

Merging 3 files ...


  0%|          | 0/3 [00:00<?, ?it/s]

ValueError: Excel file format cannot be determined, you must specify an engine manually.

In [68]:
# merge the files by looping over them and appending to an initially empty dataframe
start_time = time.time()
print(f'Merging {len(files_xls)} files ...')

# list the .xlsx files just downloaded from Eagle
files_xlsx = [fl + 'x' for fl in files_xls]

df = pd.DataFrame() # initialise empty dataframe
for f in notebook.tqdm(files_xlsx):
    data = pd.read_excel(os.path.join(pthDl, f), sheet_name = 'Reg 28 Report - Incl Effective ')
    df = pd.concat([df, data])

print(f'Merging {len(files_xls)} files completed: {timediff(start_time, time.time())}', '\n')

Merging 2 files ...


  0%|          | 0/2 [00:00<?, ?it/s]

Merging 2 files completed: 0.2sec 



In [86]:
df2

Entity Name
PRPABF    729,260,617.98
Name: End Market Value, dtype: object

In [83]:
# export dataframe to Excel
start_time = time.time()
print(f'Saving the merged files ...')

#https://towardsdatascience.com/apply-thousand-separator-and-other-formatting-to-pandas-dataframe-45f2f4c7ab01
df2 = df.groupby('Entity Name')['End Market Value'].sum().map('{:,.2f}'.format)
df2.to_excel(os.path.join(pthTest, 'Summary.xlsx') , sheet_name = 'Summary')
s = 's' if len(files_xls) != 1 else ''
lthr_name = os.path.join(pthTest, f'Lookthroughs ({len(files_xls)} fund{s}) {datetime.datetime.strftime(rptDate, "%d%b%Y")}.xlsx')
df.to_excel(lthr_name, sheet_name = 'ALL', index = False)

os.startfile(os.path.realpath(pthTest)) # open the Downloads folder

print(f'Saving the merged files completed: {timediff(start_time, time.time())}', '\n')
print(f'Roundtrip time to merge {len(files_xls)} eagle reports for {datetime.datetime.strftime(rptDate, "%d%b%Y")} completed: \
            {timediff(start_time0, time.time())}')

Saving the merged files ...
Saving the merged files completed: 2.3sec 

Roundtrip time to merge 2 eagle reports for 31Jan2025 completed:             1hr 108min 41.4sec


In [85]:
# (10) write the dataframe to review it as a workbook
start_time = time.time()
print('Writing the dataframe to a sheet for review ...')

gator = pd.ExcelWriter(lthr_name, engine = 'xlsxwriter') #!pip install xlsxwriter
fund_navs.to_excel(gator, index = False, sheet_name = 'uniques')
gator.close()

print(f'Writing the dataframe to a sheet for review completed: {timediff(start_time, time.time())}','\n')

Writing the dataframe to a sheet for review ...
Writing the dataframe to a sheet for review completed: 0.2sec 



In [80]:
fln_navs

'C:\\Users\\hilton.netta\\Downloads\\FNAV (1) 31Jan2025.xlsx'

In [ ]:
# Reg 28 & 30  P:\Investment Operations\GRC\Compliance\Reg28 and Reg30 Reporting
# folder       P:\Working Folders\Hilton\W\Reg_Tests